# Previous Sessions

61 Day 61 - Introduction to LSTM <br>
62 Day 62 - Mathematical intution of LSTM

# In this session, we will create a model which predicts next word based on previous one.

Here, we will use FAQs of CampusX which will predict next word based on this document.
The strategy is:
We will first convert it to a supervise problem with input and output and then we know how to deal with the supervised data.
Supose there are two sentences in our FAQs,
1) Hii my name is Nikunj
2) I live in India

We will create input and out like this

input               output <br>
___________________________________________<br>
Hi                   my <br>
Hii my               name <br>
Hii my name          is <br>
Hii my name is       Nikunj <br>
I                    live <br>
I live               in <br>
I live in            India <br>

Like this we will do with all the sentences.

But we know neural network does not understand categorical data. So we will have to convert this into numbers

input               output <br>
___________________________________________<br>
[1]                   2 <br>
[1, 2]                3 <br>
[1, 2, 3]             4 <br>
[1, 2, 3, 4]          5 <br>
[6]                   7 <br>
[6, 7]                8 <br>
[6, 7, 8]             9 <br>

In [14]:
print(x.shape)
print(y.shape)

(863, 56)
(863,)


In [15]:
len(tokenizer.word_index)

282

Looking at the above info, say, how many nodes we will keep in our output layers? <br>
265? But as 283 (which is the max value in y) > 265, sparse_categorical_crossentropy will raise error.  <br>

Moreove 283 is a maximum value, it means there will also be 275, 281, 280 etc... which are also greater tha 265. So all these will raise error.  <br>

Now, if we keep 283 nodes, it will simply waste resources and computational time wastage bcz we know we have only 265 unique values, othere are just repeating. V

See the below outputs,  <br>
[0, 1, 2, 1, 0, 1, 45]. Here we have 4 unique outputs 0, 1 , 2 and 45. But is it logical to keep 45 nodes in output layer just bcz there is a value 45? No.  <br>

So we should OneHotEncode this outputs using `to_categorical() which will convert these values in the form of [1, 0, 0, 0] for 0, [0, 0, 0, 1] for 45 and so on.
 <br>
 
Now, instead of 45 nodes, we can keep 4 nodes in output layer, which can save time and resources as well.
 <br>
 
Keeping 45 nodes will technically work but it will  <br>

`Problem 1`: **Wasted Output Units** <br>

This creates 45 output nodes, but you're only using 4 of them.  <br>
So 41 nodes are always ignored, and the model wastes effort computing their probabilities.

`Problem 2`: **Label Confusion** <br>
sparse_categorical_crossentropy doesn't know that labels like 42 and 283 are "class 0 and 1 in your real meaning".  <br>

It treats 283 as "class 283", i.e. it expects the 284th neuron to fire, and gradients will only update that one.  <br>
But you're really working with only 5 classes!  <br>

This makes:
* learning harder,
* convergence slower,
* and the model less accurate.



# We are using to_categorical() bcz it is used to encode numerical feature whereas OneHotEncoder is used to encode categorical feature. Out y variable is having integer data like 5, 14, 65, 165 etc, so, we will use to_categorical(). OHE is used for data like cat, dog, lion...

# So, let's apply label encoding to y (output)

In [21]:
vocab_size = len(tokenizer.word_counts)
vocab_size

282

In [22]:
from tensorflow.keras.utils import to_categorical

In [23]:
y = to_categorical(y, num_classes = 283) # 283 bcz len() counts from 1 and num_classes counts from 0, it wont consider 283rd

In [26]:
y[0]

array([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

In [27]:
y[0].shape

(283,)

# Model building

In [25]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, Dense, LSTM

In [28]:
model = Sequential()

model.add(Embedding(input_dim = 284, output_dim = 100, input_length = 56))

# input_dim = 284 means there are 284 words i.e vocab
# Output_dim = 100 means we want vector of 100 numbers and 
# input_len = 56 is the size of each sentence

model.add(LSTM(units = 150))    # LSTM layer, units is like nodes in Dense
model.add(Dense(283, activation = 'softmax'))    # units = 265 bcz we have 265 unique values in y

In [29]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 56, 100)           28400     
                                                                 
 lstm (LSTM)                 (None, 150)               150600    
                                                                 
 dense (Dense)               (None, 283)               42733     
                                                                 
Total params: 221733 (866.14 KB)
Trainable params: 221733 (866.14 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [30]:
model.compile(loss = 'categorical_crossentropy',
             optimizer = 'adam',
             metrics = ['accuracy'])

In [32]:
history = model.fit(x, y, validation_split = 0.2, epochs = 100)

# This model will overfit a lot but it is not our concern here, our concern is just learning how thinngs work here.

Epoch 1/100
22/22 [==============================] - 2s 68ms/step - loss: 0.1373 - accuracy: 0.9449 - val_loss: 7.1705 - val_accuracy: 0.2832
Epoch 2/100
22/22 [==============================] - 1s 57ms/step - loss: 0.1362 - accuracy: 0.9478 - val_loss: 7.1975 - val_accuracy: 0.2717
Epoch 3/100
22/22 [==============================] - 1s 61ms/step - loss: 0.1341 - accuracy: 0.9464 - val_loss: 7.2118 - val_accuracy: 0.2659
Epoch 4/100
22/22 [==============================] - 1s 52ms/step - loss: 0.1333 - accuracy: 0.9449 - val_loss: 7.2377 - val_accuracy: 0.2890
Epoch 5/100
22/22 [==============================] - 1s 56ms/step - loss: 0.1323 - accuracy: 0.9435 - val_loss: 7.1957 - val_accuracy: 0.2832
Epoch 6/100
22/22 [==============================] - 1s 63ms/step - loss: 0.1307 - accuracy: 0.9478 - val_loss: 7.2358 - val_accuracy: 0.2832
Epoch 7/100
22/22 [==============================] - 1s 62ms/step - loss: 0.1313 - accuracy: 0.9449 - val_loss: 7.2699 - val_accuracy: 0.2659
Epoch 

In [43]:


# Making prediction

text = 'amount'

# tokennize
tokenized_text = tokenizer.texts_to_sequences([text])[0]

# padding
padded_text = pad_sequences([tokenized_text], maxlen = 56, padding = 'pre')

# Predicting

pred = model.predict(padded_text)

# Getting the index of max prediction
pred_index = np.argmax(pred)

for word, index in tokenizer.word_index.items():
    if index == pred_index:
        print(word)

1/1 [==============================] - 0s 104ms/step
of


In [45]:
np.argmax(pred)

6

In [37]:
# Let's predict next 5 words

text = 'amount'

for i in range(5):

    # tokennize
    tokenized_text = tokenizer.texts_to_sequences([text])[0]

    # padding
    padded_text = pad_sequences([tokenized_text], maxlen = 56, padding = 'pre')

    # Predicting
    pred = model.predict(padded_text)

    # Getting the index of max prediction
    pred_index = np.argmax(pred)

    for word, index in tokenizer.word_index.items():
        if index == pred_index:
            text = text + ' ' + word
            print(text)


1/1 [==============================] - 0s 37ms/step
amount of
1/1 [==============================] - 0s 46ms/step
amount of all
1/1 [==============================] - 0s 27ms/step
amount of all the
1/1 [==============================] - 0s 25ms/step
amount of all the live
1/1 [==============================] - 0s 29ms/step
amount of all the live sessions


# We may not get proper word predictions but it is not at all our concern. We just wanted to learn how things work in LSTM

In [57]:
text = 'mail'

for i in range(10):
    # Tokenize
    tokenized_text = tokenizer.texts_to_sequences([text])[0]

    # Pad
    padded_text = pad_sequences([tokenized_text], maxlen=56, padding='pre')

    # Predict
    pred = model.predict(padded_text, verbose=0)
    pred_index = np.argmax(pred)

    for word, index in tokenizer.word_index.items():
        if index == pred_index:
            text = text + ' ' + word
            print(text)


mail us
mail us by
mail us by sending
mail us by sending a
mail us by sending a mail
mail us by sending a mail at
mail us by sending a mail at nitish
mail us by sending a mail at nitish campusx
mail us by sending a mail at nitish campusx gmail
mail us by sending a mail at nitish campusx gmail com


# Tips to improve the model's performance.

We can see the model is overfitting a lot as there is a great difference between training and val_acccuracy and also not predicting words accurately. So, we can use below given steps to overcome overfitting and imporve the working of the model.

1) More data
2) We can use more LSTM layers.
3) Can do hyperparameter tunning.
4) Use Bi-directional LSTMs `(Will learn in our later sessions)`.
5) Use transformers; like ChatGPT, BERT etc `(Will learn in our later sessions)`.